# 不加任何修饰的最纯的Self-Attention
论文就是经典的Attention is All You Need，链接我也放上来 https://arxiv.org/abs/1706.03762# 还是推荐读一读这篇论文以示尊敬。

简单先放一个图在这里，但是并不实现他，先有个概念：

左边是Encoder编码用的；     右边是Decoder解码用的；     下面是embedding把文本变成特征；     上面是lm-head再把特征变成文本。

Encoder和Decoder里有很多的Attention，这个Attention就是transformer的核心，也是这个notebook实现的东西，至于其他的先不用管。

<img src='./figure/transformers.png' width="30%">

这个notebook里我们不做完整的transformer，只实现一个Attention，那首先就定一个Attention类，我假定看到这里的人对torch都已经有了基本了解。

# Attention实现

In [1]:
import torch

# 一般来说，nlp任务的input形式都是(B,L,D)，代表的啥意思英文已经说明白了。
batch_size = 2
seq_length = 4
hidden_state_dims = 8
x = torch.randn(batch_size, seq_length, hidden_state_dims)

class Attention(torch.nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, x):
        output = x
        return output

attention = Attention()
y = attention(x)

#搞清楚流程最简单的办法就是打印数据的shape
print('x的shape：{}'.format(x.shape))   
print('y的shape：{}'.format(y.shape))   

#作为一个Attention类，输入输出的shape应该是一样的，不一样就出问题了。
assert x.shape==y.shape

#下面这句可以考虑不注释看看会爆什么错
# assert x.shape!=y.shape            

x的shape：torch.Size([2, 4, 8])
y的shape：torch.Size([2, 4, 8])


## 简单介绍一下attention是个什么玩意儿

对于transformer（尤其是现在纯decoder结构的）来说，我们把他叫做一个自回归模型

我们每输入一个token它就会把这个token的信息缓存住同时输出一个token，我们把这个输出的token再输入进来他就又缓存又输出循环往复。

## Attention具体怎么实现的呢？

我们有三个矩阵$Q、K、V$，就是query，key和value

Q矩阵的大小都是$(B,1,D)$，为什么一维是1呢（维度从左往右从零维开始）？，我们认为这个这就是上面说的我们当前输入的token。

K、V矩阵的大小是$(B,L,D)$，为什么一维是$L$呢？这个就是上面说的我前面缓存住的L个token。

具体流程论文里是放的是这个图：

<img src='./figure/normal_attention.png' width="15%">

那些scale和mask先不用管，就当他是一些操作，后面会展开

现在需要知道就是Q和K做一些操作再softmax后会得到一个$(B,L)$的权重矩阵，就是说这个Q更关注K的哪些部分，然后权重毕竟是个比例而比例的量纲是1因此必须去加权平均一个别的东西才有意义，这个加权平均的东西就是V

我们结合一个具体的例子来看

## 看一眼这个公式 $Attention(Q, K, V)=softmax(\frac{QK^{T}}{\sqrt{d_k}})V$

上面说了，Q是$(B,1,D)$，K和V是$(B,L,D)$

那么$QK^T$就是$(B,1,L)$，那么$softmax(\frac{QK^{T}}{\sqrt{d_k}})$的shape就是$(B,1,D) \times (B,L,D)=(B,1,L)$，这一步得到的就是一个Q和所有K得匹配结果，或者说：每“1”个Q对每“L”个位置的关注程度。

那么$softmax(\frac{QK^{T}}{\sqrt{d_k}})V$的shape就是$(B,1,L) \times (B,L,D)=(B,1,D)$，这一步就是把匹配结果给加权到每一个权重V上。

## $(B,1,D)$还是谁的shape？    Q的。

这就说明了$Attention(Q,K,V)$的意义就是从Q出发，对所有已知量K去匹配，再把匹配的结果作为权重去加权平均每一个V，最后必然得到一个和Q形状一样的东西。

## $\sqrt{d_k}$是什么呢？

是一个系数，$d_k$就是特征维度，因为很明显$q \cdot k = \sum_{i=1}^{d_k} q_i k_i$这个数字可能会很大毕竟$d_k$数相加后面还要做一个softmax，那除以一个数字很合理吧

## 为什么除的是$\sqrt{d_k}$？

我们假设$Q$ 和 $K$ 的分量都是相互独立的随机变量，且满足均值为 0、方差为 1（即 $E[q]=0, Var(q)=1$），那么$E(q \cdot k)=0, 但是Var(q \cdot k)=d_k$

方差$d_k$啊这能忍吗？显然不能，so除以一个$\sqrt{d_k}$就变成0-1分布了就很ok了。

In [2]:
# 手动搞一下我们的QKV
# 简单点Q的shape是[2, 1, 3]
Q = torch.tensor([[[1, 2, 3]]]*2, dtype=torch.bfloat16)     
# KV的shape都是[2, 2, 3], K是故意设计成这样的，第一个是Q的线性变化，第二个和Q的内积是0
K = torch.tensor([[[2, 4, 6], [-3, 0, 1]]]*2, dtype=torch.bfloat16)
V = torch.rand_like(K)

# 用matmul也可以，但我更强倾向于用einsum，回过头来看的时候能确切知道自己在做什么。
attention_weights_matmal = torch.matmul(Q, K.transpose(-1, -2))
attention_weights = torch.einsum('bqd, bLd->bqL', Q, K)

print("得到的weight的样子:\n", attention_weights, "\n")
print("*"*50)
print("weight的shape:\n", attention_weights.shape, "\n")
print(f"表示对于一个大小为{attention_weights.shape[0]}的batch, 其每{attention_weights.shape[1]}个查询向量对应缓存的{attention_weights.shape[2]}个向量的相似度。")
print("*"*50)
print("两种方式得到的attention是否相等:\n", attention_weights == attention_weights_matmal)

得到的weight的样子:
 tensor([[[28.,  0.]],

        [[28.,  0.]]], dtype=torch.bfloat16) 

**************************************************
weight的shape:
 torch.Size([2, 1, 2]) 

表示对于一个大小为2的batch, 其每1个查询向量对应缓存的2个向量的相似度。
**************************************************
两种方式得到的attention是否相等:
 tensor([[[True, True]],

        [[True, True]]])


In [3]:
d_k = torch.tensor(Q.shape[-1])

attention_weights = torch.softmax(attention_weights/torch.sqrt(d_k), dim=-1)

# 可以看出来，基本权重都在第一个位置上，说明希望关注第一个位置的V
print("最后得到的注意力权重:\n", attention_weights)

最后得到的注意力权重:
 tensor([[[1.0000e+00, 9.9186e-08]],

        [[1.0000e+00, 9.9186e-08]]], dtype=torch.bfloat16)


In [4]:
# 同样的，我还是希望用einsum
output = torch.einsum("bqL, bLd->bqd", attention_weights, V)

#可以看出来由于只关注第1个位置的信息，因此相当于完全copy了V里两个batch里的第一条数据。
print(output)
print("*"*50)
print(V)
print("*"*50)
print("Q和output的shape一样吗？\n", Q.shape==output.shape)

tensor([[[0.4883, 0.0469, 0.2500]],

        [[0.3086, 0.1133, 0.8242]]], dtype=torch.bfloat16)
**************************************************
tensor([[[0.4883, 0.0469, 0.2500],
         [0.1367, 0.8750, 0.2500]],

        [[0.3086, 0.1133, 0.8242],
         [0.4297, 0.1875, 0.4531]]], dtype=torch.bfloat16)
**************************************************
Q和output的shape一样吗？
 True


In [5]:
#把上面的过程封装为一个函数
def attention_function(Q, K, V):
    d_k = torch.tensor(Q.shape[-1])
    attention_weights = torch.einsum('bqd, bLd->bqL', Q, K)
    attention_weights = torch.softmax(attention_weights/torch.sqrt(d_k), dim=-1)
    output = torch.einsum("bqL, bLd->bqd", attention_weights, V)
    #相信你也看出来了，这个bqL里的q取什么值这个attention都能运行，这里埋个伏笔在下个notebook里讲。
    return output
#至此恭喜你学会了最简单的attention

In [6]:
#然后我们再把这个函数集合到前面的Attention类里
class Attention(torch.nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
    
    def forward(self, Q, K, V):
        d_k = torch.tensor(Q.shape[-1])
        attention_weights = torch.einsum('bqd, bLd->bqL', Q, K)
        attention_weights = torch.softmax(attention_weights/torch.sqrt(d_k), dim=-1)
        output = torch.einsum("bqL, bLd->bqd", attention_weights, V)
        return output

## 能不能看出这个class存在相当多的问题？

很显然：

1、d_k每次都计算一次这很不合理，肯定应该是在__init__里就应该实现的。

2、对于user来说肯定是输入越简单越好Q、K、V也太复杂了，现在的forward需要输入3个矩阵，但是前面我们说了其实只需要输入一个token。

****从封装上看，我们希望能把从token到Q、K、V的过程放在Attention类中进行，这样在不同Attention就可以一个for直连，代码就很简洁了不需要额外处理。

我在[第二节](02.ipynb)中解决这两问题。
